# 🚀 FER Projesi - Profesyonel Google Drive Destekli SOTA Model Eğitimi (%80+ Hedefi)
Bu not defteri modern SOTA mimarileri (**ViT-Base**, **ConvNeXt-Tiny**), **ArcFace Loss** ve **AffectNet + RAF-DB + FER2013** 3'lü devasa veri kümesiyle Colab GPU üzerinde kesintisiz çalışır. **Google Drive bağlantısı sayesinde elektrik kesilse veya bağlantı kopsa bile kaldığınız checkpoint'ten otomatik devam eder!**

In [ ]:
# 1. Google Drive'ı Otomatik Bağlama & Yedek Klasörü Hazırlığı
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_BACKUP = '/content/drive/MyDrive/fer_backup'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
print(f'✅ Google Drive başarıyla bağlandı! Yedek klasörü: {DRIVE_BACKUP}')

In [ ]:
# 2. Ekran Kartı (GPU) Kontrolü ve Kütüphane Kurulumu
!nvidia-smi
!pip install timm albumentations dacite onnxruntime-gpu gradio opencv-python-headless scikit-learn kagglehub pandas kaggle

In [ ]:
# 3. Proje ZIP Dosyasını Yükleme ve Çıkarma
from google.colab import files
import zipfile, os

if not os.path.exists('src'):
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as zip_ref:
                zip_ref.extractall('.')

if os.path.exists('fer-project'):
    %cd fer-project
print('✅ Proje ortamı hazır! Çalışma dizini:', os.getcwd())

In [ ]:
# 4. AffectNet Veri Setini Kaggle API İle Hızlı İndirme (eyllkalfa)
import os, json

os.environ['KAGGLE_USERNAME'] = 'eyllkalfa'
os.environ['KAGGLE_KEY'] = 'KGAT_353cfed36b7416845c6e09a5307fd7a1'

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': 'eyllkalfa', 'key': 'KGAT_353cfed36b7416845c6e09a5307fd7a1'}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

os.makedirs('data/raw/affectnet', exist_ok=True)
!kaggle datasets download -d mstjebashazida/affectnet -p data/raw/affectnet --unzip || true
print('✅ AffectNet indirme işlemi başarıyla tamamlandı!')

In [ ]:
# 5. Tüm Veri Setlerini İşleme (RAF-DB + FER2013 + AffectNet 100.000+ Resim)
!PYTHONPATH=src python scripts/preprocess_dataset.py --dataset rafdb --bypass-face-detection
!PYTHONPATH=src python scripts/preprocess_dataset.py --dataset fer2013 --bypass-face-detection
!PYTHONPATH=src python scripts/preprocess_dataset.py --dataset affectnet --bypass-face-detection || true

In [ ]:
# 6. SOTA Model Eğitimi + Otomatik Google Drive Yedekleme & Kaldığı Yerden Devam Etme
import glob, shutil, os

DRIVE_BACKUP = '/content/drive/MyDrive/fer_backup'
drive_ckpts = sorted(glob.glob(f'{DRIVE_BACKUP}/experiments/*/checkpoints/best_model.pt'))
resume_flag = f'--resume {drive_ckpts[-1]}' if drive_ckpts else ''

if resume_flag:
    print(f'🔄 Önceki checkpoint bulundu! Kaldığı yerden devam ediliyor: {drive_ckpts[-1]}')
else:
    print('🚀 Yeni ViT-Base + ArcFace SOTA Eğitimi Başlatılıyor...')

# Eğitimi Başlat
!PYTHONPATH=src python scripts/train.py --config configs/base.yaml configs/vit_sota.yaml {resume_flag}

# Eğitim bittiğinde veya durduğunda otomatik Drive'a kopyala
print('📦 Deney sonuçları Google Drive üzerine kopyalanıyor...')
if os.path.exists('experiments'):
    shutil.copytree('experiments', os.path.join(DRIVE_BACKUP, 'experiments'), dirs_exist_ok=True)
print('🎉 Drive yedeklemesi %100 tamamlandı!')

In [ ]:
# 7. Model Değerlendirmesi ve Sıcaklık Kalibrasyonu
import glob
ckpts = sorted(glob.glob('experiments/*/checkpoints/best_model.pt')) + sorted(glob.glob('/content/drive/MyDrive/fer_backup/experiments/*/checkpoints/best_model.pt'))
if ckpts:
    best_ckpt = ckpts[-1]
    !PYTHONPATH=src python scripts/evaluate.py --config configs/base.yaml configs/vit_sota.yaml --checkpoint {best_ckpt} --calibrate

In [ ]:
# 8. 📊 TÜM DENENEN MODELLERİ OTOMATİK KARŞILAŞTIRMA VE KAYDETME
!PYTHONPATH=src python scripts/compare_experiments.py
# Karşılaştırma tablosunu da Drive'a kopyalayalım
if os.path.exists('exports'):
    shutil.copytree('exports', os.path.join(DRIVE_BACKUP, 'exports'), dirs_exist_ok=True)

In [ ]:
# 9. Eğitilmiş Modelin ONNX Formatına Dönüştürülmesi
if ckpts:
    !PYTHONPATH=src python scripts/export_onnx.py --config configs/base.yaml configs/vit_sota.yaml --checkpoint {best_ckpt} --output exports/mobilenetv3.onnx

In [ ]:
# 10. Canlı Web Demosunun Başlatılması (Canlı Bağlantı Linki Üretir)
!PYTHONPATH=src python scripts/demo.py --model exports/mobilenetv3.onnx --share